In [10]:
import pandas as pd
df_hackathon = pd.read_csv("dataset/transactions_mock_1000_for_participants.csv")
df = pd.read_csv("dataset/AML_dataset.csv")

In [9]:
df_hackathon.columns

Index(['transaction_id', 'booking_jurisdiction', 'regulator',
       'booking_datetime', 'value_date', 'amount', 'currency', 'channel',
       'product_type', 'originator_name', 'originator_account',
       'originator_country', 'beneficiary_name', 'beneficiary_account',
       'beneficiary_country', 'swift_mt', 'ordering_institution_bic',
       'beneficiary_institution_bic', 'swift_f50_present', 'swift_f59_present',
       'swift_f70_purpose', 'swift_f71_charges', 'travel_rule_complete',
       'fx_indicator', 'fx_base_ccy', 'fx_quote_ccy', 'fx_applied_rate',
       'fx_market_rate', 'fx_spread_bps', 'fx_counterparty', 'customer_id',
       'customer_type', 'customer_risk_rating', 'customer_is_pep',
       'kyc_last_completed', 'kyc_due_date', 'edd_required', 'edd_performed',
       'sow_documented', 'purpose_code', 'narrative', 'is_advised',
       'product_complex', 'client_risk_profile', 'suitability_assessed',
       'suitability_result', 'product_has_va_exposure',
       'va_dis

In [12]:
df.columns

Index(['Time', 'Date', 'Sender_account', 'Receiver_account', 'Amount',
       'Payment_currency', 'Received_currency', 'Sender_bank_location',
       'Receiver_bank_location', 'Payment_type', 'Is_laundering'],
      dtype='object')

In [13]:
import pandas as pd
import numpy as np
import os

def preprocess_hackathon_data(input_file, output_file):
    """
    Loads the hackathon data, renames/engineers columns to match our
    V2 training data format, and saves a new 'cleaned' CSV.
    """
    print(f"Loading hackathon data from '{input_file}'...")
    if not os.path.exists(input_file):
        print(f"Error: File not found at {input_file}")
        return

    df = pd.read_csv(input_file)
    print("Original columns:", df.columns.tolist())

    # --- 1. Define the "Adapter" Map ---
    # Maps (Their Column Name) -> (Our Column Name)
    column_map = {
        'originator_account': 'Sender_account',
        'beneficiary_account': 'Receiver_account',
        'amount': 'Amount',
        'booking_datetime': 'Timestamp', # We'll create Date/Time from this
        'originator_country': 'Sender_bank_location',
        'beneficiary_country': 'Receiver_bank_location',
        'channel': 'Payment_type',
        # We can map other columns if our V2 script uses them,
        # but these are the essentials.
    }

    # Rename columns
    df.rename(columns=column_map, inplace=True)

    # --- 2. Engineer the Label ---
    # Convert 'str_filed_datetime' to a 1/0 label
    # We assume 'not null' means it IS laundering
    if 'str_filed_datetime' in df.columns:
        df['Is_laundering'] = np.where(df['str_filed_datetime'].notnull(), 1, 0)
    else:
        # If no label, assume all are 0 (for final test data)
        df['Is_laundering'] = 0

    # --- 3. Engineer Date and Time columns ---
    # Our build_graph_v2.py script *expects* 'Date' and 'Time'
    if 'Timestamp' in df.columns:
        print("Parsing 'Timestamp' into 'Date' and 'Time'...")
        df['Timestamp'] = pd.to_datetime(df['Timestamp'], errors='coerce')
        df['Date'] = df['Timestamp'].dt.strftime('%Y-%m-%d')
        df['Time'] = df['Timestamp'].dt.strftime('%H:%M:%S')
    else:
        print("Error: Could not find 'booking_datetime' (renamed to 'Timestamp').")
        return

    # --- 4. Filter for *Only* the columns our script needs ---
    # This drops all the extra "golden" features we can't use
    required_columns = [
        'Time', 'Date', 'Sender_account', 'Receiver_account', 'Amount',
        'Payment_currency', 'Received_currency', 'Sender_bank_location',
        'Receiver_bank_location', 'Payment_type', 'Is_laundering'
    ]
    
    # Handle columns that might not exist in their data but are in ours
    # (e.g., Payment_currency, Received_currency)
    for col in required_columns:
        if col not in df.columns:
            print(f"Warning: Column '{col}' not found. Filling with 'NULL'.")
            # We map 'currency' to 'Payment_currency' as a fallback
            if col == 'Payment_currency' and 'currency' in df.columns:
                df['Payment_currency'] = df['currency']
            # We can't know received, so just copy payment
            elif col == 'Received_currency' and 'Payment_currency' in df.columns:
                 df['Received_currency'] = df['Payment_currency']
            else:
                df[col] = 'NULL' # Create a placeholder

    # Filter for *only* these columns
    df_cleaned = df[required_columns]

    # --- 5. Save the Cleaned File ---
    df_cleaned.to_csv(output_file, index=False)
    print(f"\nSuccess! Cleaned data saved to '{output_file}'")
    print(f"  - Renamed columns to match our format.")
    print(f"  - Engineered 'Is_laundering' label.")
    print(f"  - Dropped {len(df.columns) - len(required_columns)} unused columns.")
    print("You are now ready to run 'build_graph_v2.py' on this new file.")


In [14]:
# --- Configuration ---
# 1. Put the hackathon's file in your 'dataset/' folder
HACKATHON_FILE_PATH = 'dataset/transactions_mock_1000_for_participants.csv' 

# 2. This is the new file we will create
CLEANED_FILE_PATH = 'dataset/hackathon_sample_CLEANED.csv'
# ---

preprocess_hackathon_data(HACKATHON_FILE_PATH, CLEANED_FILE_PATH)

Loading hackathon data from 'dataset/transactions_mock_1000_for_participants.csv'...
Original columns: ['transaction_id', 'booking_jurisdiction', 'regulator', 'booking_datetime', 'value_date', 'amount', 'currency', 'channel', 'product_type', 'originator_name', 'originator_account', 'originator_country', 'beneficiary_name', 'beneficiary_account', 'beneficiary_country', 'swift_mt', 'ordering_institution_bic', 'beneficiary_institution_bic', 'swift_f50_present', 'swift_f59_present', 'swift_f70_purpose', 'swift_f71_charges', 'travel_rule_complete', 'fx_indicator', 'fx_base_ccy', 'fx_quote_ccy', 'fx_applied_rate', 'fx_market_rate', 'fx_spread_bps', 'fx_counterparty', 'customer_id', 'customer_type', 'customer_risk_rating', 'customer_is_pep', 'kyc_last_completed', 'kyc_due_date', 'edd_required', 'edd_performed', 'sow_documented', 'purpose_code', 'narrative', 'is_advised', 'product_complex', 'client_risk_profile', 'suitability_assessed', 'suitability_result', 'product_has_va_exposure', 'va_disc

In [25]:
df = pd.read_csv("dataset/hackathon_sample_CLEANED.csv")

In [26]:
df['Payment_type'].unique()

array(['RTGS', 'SWIFT', 'FAST (SG)', 'SEPA', 'Cash', 'FPS (HK)', 'Cheque'],
      dtype=object)

In [20]:
df

,Time,Date,Sender_account,Receiver_account,Amount,Payment_currency,Received_currency,Sender_bank_location,Receiver_bank_location,Payment_type,Is_laundering
0,10:24:43,2024-10-10,GB91GCCI85456233719196,GB80LAUP42959382164081,590012.92,HKD,HKD,AE,CN,RTGS,0
1,23:56:23,2024-02-23,GB39OOLA52427580832378,GB88KUDJ48147748190437,1319007.62,GBP,GBP,TH,CN,SWIFT,0
2,23:40:37,2024-06-26,GB61CLBF11432797139540,GB75GNZJ38259031799913,233935.30,GBP,GBP,VN,CH,FAST (SG),0
3,00:08:29,2024-05-03,GB56TQOT03483189875583,GB19HQGX56751391902097,1778002.31,GBP,GBP,AU,US,SEPA,0
4,13:58:10,2024-12-03,GB30TZUH89833514426746,GB60ORBY07341123489694,2479099.40,HKD,HKD,HK,DE,RTGS,0
...,...,...,...,...,...,...,...,...,...,...,...
995,12:25:08,2024-10-24,GB92OWRQ10413262781678,GB19UPEE05133828734875,7704.06,USD,USD,HK,GB,Cash,0
996,10:28:21,2025-01-15,GB24MECU82880339781077,GB18QFDK12697227968909,7505.22,HKD,HKD,GB,VN,Cheque,0
997,03:47:21,2024-08-21,GB48EKXO11446580042573,GB14PZOX23487683347626,1274.27,SGD,SGD,RU,DE,Cash,0
998,04:30:50,2025-01-11,GB54MCIU36705490640083,GB62TAWE02132771030635,743971.25,HKD,HKD,FR,IR,Cheque,0


In [21]:
df = pd.read_csv("dataset/AML_dataset.csv")
df

,Time,Date,Sender_account,Receiver_account,Amount,Payment_currency,Received_currency,Sender_bank_location,Receiver_bank_location,Payment_type,Is_laundering
0,10:35:19,2022-10-07,8724731955,2769355426,1459.15,UK pounds,UK pounds,UK,UK,Cash Deposit,0
1,10:35:20,2022-10-07,1491989064,8401255335,6019.64,UK pounds,Dirham,UK,UAE,Cross-border,0
2,10:35:20,2022-10-07,287305149,4404767002,14328.44,UK pounds,UK pounds,UK,UK,Cheque,0
3,10:35:21,2022-10-07,5376652437,9600420220,11895.00,UK pounds,UK pounds,UK,UK,ACH,0
4,10:35:21,2022-10-07,9614186178,3803336972,115.25,UK pounds,UK pounds,UK,UK,Cash Deposit,0
...,...,...,...,...,...,...,...,...,...,...,...
9504847,10:57:01,2023-08-23,2453933570,519744068,2247.25,UK pounds,UK pounds,UK,UK,ACH,0
9504848,10:57:06,2023-08-23,9805510177,5416607878,927.18,UK pounds,UK pounds,UK,UK,Debit card,0
9504849,10:57:06,2023-08-23,7282330957,2995527149,1455.14,UK pounds,UK pounds,UK,UK,ACH,0
9504850,10:57:11,2023-08-23,940337377,4812815165,25995.70,UK pounds,UK pounds,UK,UK,ACH,0


In [24]:
df['Payment_type'].unique()

array(['Cash Deposit', 'Cross-border', 'Cheque', 'ACH', 'Credit card',
       'Debit card', 'Cash Withdrawal'], dtype=object)